# Exploratory Data Analysis: CGM Glucose Patterns & Hormonal Cycle Mapping
This notebook analyzes 142,511 CGM readings (LinX) from June to October 2026, mapped against actual menstrual cycle dates to uncover how hormonal shifts (estrogen, progesterone, LH surge, and cycle delay/amenorrhea) influence glucose metabolism.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Load the processed dataset with mapped menstrual cycle attributes
df = pd.read_csv('../data/processed/glucose_with_cycle.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = pd.to_datetime(df['date'])

print(f'Total 5-minute CGM readings: {len(df):,}')
print(f'Date range: {df["timestamp"].min()} to {df["timestamp"].max()}')
df.head()

## 1. Clinical CGM Summary Metrics
Standard clinical consensus metrics (ADA / ATTD guidelines) for Continuous Glucose Monitoring.

In [ ]:
mean_g = df['glucose'].mean()
std_g = df['glucose'].std()
cv_g = (std_g / mean_g) * 100
median_g = df['glucose'].median()
gmi = 3.31 + (0.02392 * mean_g)

total_pts = len(df)
tir = (df['glucose'].between(70, 180)).sum() / total_pts * 100
tir_tight = (df['glucose'].between(70, 140)).sum() / total_pts * 100
tbr_very_low = (df['glucose'] < 54).sum() / total_pts * 100
tbr_low = (df['glucose'].between(54, 69.999)).sum() / total_pts * 100
tar_high = (df['glucose'].between(180.001, 250)).sum() / total_pts * 100
tar_very_high = (df['glucose'] > 250).sum() / total_pts * 100

pd.DataFrame({
    'Metric': [
        'Mean Glucose', 'Median Glucose', 'Standard Deviation (SD)',
        'Coefficient of Variation (CV%)', 'GMI (Estimated HbA1c)',
        'Time in Range (70-180 mg/dL)', 'Tight Time in Range (70-140 mg/dL)',
        'Time Below Range (<70 mg/dL)', 'Time Very Low (<54 mg/dL)',
        'Time Above Range (>180 mg/dL)', 'Time Very High (>250 mg/dL)'
    ],
    'Value': [
        f'{mean_g:.1f} mg/dL', f'{median_g:.1f} mg/dL', f'{std_g:.1f} mg/dL',
        f'{cv_g:.1f}%', f'{gmi:.2f}%',
        f'{tir:.1f}%', f'{tir_tight:.1f}%',
        f'{(tbr_low + tbr_very_low):.1f}%', f'{tbr_very_low:.1f}%',
        f'{(tar_high + tar_very_high):.1f}%', f'{tar_very_high:.1f}%'
    ],
    'Target Benchmark': [
        '< 154 mg/dL', '-', '-',
        '< 36% (Low variability)', '< 7.0%',
        '> 70%', '> 50%',
        '< 4%', '< 1%',
        '< 25%', '< 5%'
    ]
})

## 2. Ambulatory Glucose Profile (AGP) - 24-Hour Diurnal Pattern
Aggregates across all days to reveal the 24-hour diurnal rhythm: overnight baseline, dawn rise, meal spikes, and late evening elevation.

In [ ]:
df['hour'] = df['timestamp'].dt.hour
df['minute'] = df['timestamp'].dt.minute
df['time_dec'] = df['hour'] + df['minute'] / 60.0

time_bins = np.linspace(0, 24, 97)
df['time_bin'] = pd.cut(df['time_dec'], bins=time_bins, labels=time_bins[:-1])
agp = df.groupby('time_bin', observed=False)['glucose'].agg([
    lambda x: x.quantile(0.10),
    lambda x: x.quantile(0.25),
    'median',
    lambda x: x.quantile(0.75),
    lambda x: x.quantile(0.90)
]).rename(columns={'<lambda_0>': 'p10', '<lambda_1>': 'p25', '<lambda_2>': 'p75', '<lambda_3>': 'p90'})
agp.index = agp.index.astype(float)

plt.figure(figsize=(12, 5))
plt.fill_between(agp.index, agp['p10'], agp['p90'], color='#6baed6', alpha=0.3, label='10th - 90th Percentile')
plt.fill_between(agp.index, agp['p25'], agp['p75'], color='#2171b5', alpha=0.4, label='25th - 75th Percentile (IQR)')
plt.plot(agp.index, agp['median'], color='#08306b', linewidth=2.5, label='Median Glucose')

plt.axhline(180, color='crimson', linestyle='--', alpha=0.7, label='Upper Target (180 mg/dL)')
plt.axhline(70, color='darkorange', linestyle='--', alpha=0.7, label='Lower Target (70 mg/dL)')
plt.axhspan(70, 180, color='green', alpha=0.08, label='Target Range')

plt.title('Ambulatory Glucose Profile (AGP)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Hour of Day', fontsize=12)
plt.ylabel('Glucose (mg/dL)', fontsize=12)
plt.xticks(range(0, 25, 2), [f'{h:02d}:00' for h in range(0, 25, 2)])
plt.xlim(0, 24)
plt.ylim(40, 320)
plt.legend(loc='upper left', frameon=True)
plt.show()

## 3. Menstrual Cycle Phase Mapping & Hormonal Association
The user tracking record:
- May 27 (Duration: 4 days)
- June 22 (Duration: 4 days, Cycle length: 26 days)
- July 17 (Duration: 4 days, Cycle length: 25 days)
- Aug 19 (Duration: 4 days, Cycle length: 33 days)
- September: Skipped (Amenorrhea / extended cycle)
- Oct 31 (Duration: 4 days, Cycle length: 73 days)

Below, we examine regular cycles (Days 1–28) vs. the September skipped cycle surge.

In [ ]:
# Regular Cycles (Cycle 2 & 3) Trajectory across Cycle Days
reg = df[df['cycle_id'].isin([2, 3])].groupby('cycle_day')['glucose'].agg(['mean', 'std', 'median', 'count']).reset_index()
reg = reg[reg['count'] >= 100]

plt.figure(figsize=(12, 5))
plt.plot(reg['cycle_day'], reg['median'], 'o-', color='#e7298a', linewidth=2.5, markersize=6, label='Median Glucose')
plt.plot(reg['cycle_day'], reg['mean'], 's--', color='#7570b3', linewidth=1.5, alpha=0.7, label='Mean Glucose')

plt.axvspan(1, 4, color='#fbb4ae', alpha=0.35, label='Menstrual (Days 1-4)')
plt.axvspan(5, 11, color='#b3cde3', alpha=0.3, label='Follicular (Days 5-11)')
plt.axvspan(12, 14, color='#ccebc5', alpha=0.45, label='Ovulatory Window (Days 12-14)')
plt.axvspan(15, 26, color='#decbe4', alpha=0.3, label='Luteal Phase (Days 15+)')

plt.title('Regular Cycles: Glucose Across Menstrual Cycle Days', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Cycle Day (Day 1 = First Day of Menstrual Flow)', fontsize=12)
plt.ylabel('Blood Glucose (mg/dL)', fontsize=12)
plt.xticks(range(1, 28))
plt.legend(loc='lower left', frameon=True)
plt.show()

## 4. Phase-by-Phase Comparison Table
Detailed metrics across Menstrual, Follicular, Ovulatory, and Luteal phases.

In [ ]:
phase_stats = df.groupby('phase')['glucose'].agg([
    'count', 'mean', 'median', 'std',
    lambda x: (x.std() / x.mean()) * 100,
    lambda x: (x.between(70, 180)).mean() * 100,
    lambda x: (x > 180).mean() * 100,
    lambda x: (x < 70).mean() * 100
]).rename(columns={'<lambda_0>': 'CV%', '<lambda_1>': 'TIR (70-180)%', '<lambda_2>': 'TAR (>180)%', '<lambda_3>': 'TBR (<70)%'})
phase_stats

## 5. The September Skipped Cycle: Hormonal Impact on Glycemic Control
When the expected period was missed in mid-September (Weeks 4-5 post Aug 19 menses), glucose experienced sustained resistance peaking above 200 mg/dL.

In [ ]:
daily_all = df.groupby('date')['glucose'].agg(['mean', 'median']).reset_index()

period_dates = [pd.to_datetime(d) for d in ['2026-05-27', '2026-06-22', '2026-07-17', '2026-08-19']]

plt.figure(figsize=(14, 5))
plt.plot(daily_all['date'], daily_all['median'], color='#2b8cbe', linewidth=2, label='Daily Median Glucose')

for p in period_dates:
    plt.axvline(p, color='#e41a1c', linestyle='-', linewidth=2, alpha=0.7)

exp_sept = pd.to_datetime('2026-09-16')
plt.axvspan(exp_sept - pd.Timedelta(days=3), exp_sept + pd.Timedelta(days=10), color='#fed976', alpha=0.45, label='September Due Window (Skipped)')
plt.axhline(180, color='red', linestyle='--', alpha=0.5, label='180 mg/dL Target Threshold')

plt.title('Longitudinal Glucose Trajectory & The September Skipped Period Surge', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Blood Glucose (mg/dL)', fontsize=12)
plt.legend(loc='upper right', frameon=True)
plt.show()

## 6. Psychosocial, Academic, & Physiological Stress Mapping
Between August 26 and early October, distinct acute stress events occurred:
1. **Aug 26–29:** Hackathon with friends, interpersonal conflict, high physical exertion, and a big fight at 1:00 AM on August 29.
2. **Aug 30 – Sep 4:** Post-hackathon fallout & high anxiety.
3. **Sep 13–17:** Severe acute exam panic (fear of failing). Notice: this coincided exactly with the expected period due date, triggering functional amenorrhea (skipped period).
4. **Sep 18 – Oct 5:** Chronic anticipatory anxiety waiting for exam results.
5. **Oct 6 onwards:** Stress resolution / relief and gradual hormonal recovery leading to the October 31 period.

In [ ]:
# Compare metrics across stress events
stress_order = [
    'Baseline (June - Aug 25)',
    'Hackathon & Conflict (Aug 26-29)',
    'Exam Panic / Period Due (Sep 13-17)',
    'Exam Result Anxiety (Sep 18-Oct 5)',
    'Post-Stress Recovery (Oct 6+)'
]

stress_summary = df[df['stress_event'].isin(stress_order)].groupby('stress_event')['glucose'].agg([
    'count', 'mean', 'median', 'std',
    lambda x: (x.between(70, 180)).mean()*100,
    lambda x: (x > 180).mean()*100,
    lambda x: (x > 250).mean()*100
]).rename(columns={'<lambda_0>': 'TIR (70-180)%', '<lambda_1>': 'TAR (>180)%', '<lambda_2>': 'Very High (>250)%'})

stress_summary.reindex(stress_order)

In [ ]:
# High-resolution CGM Trace of the August 28-29 Hackathon Fight
h_trace = df[(df['timestamp'] >= '2026-08-28 18:00') & (df['timestamp'] <= '2026-08-29 13:00')].copy()

plt.figure(figsize=(12, 5))
plt.plot(h_trace['timestamp'], h_trace['glucose'], color='#d95f02', linewidth=2.5, marker='.', markersize=4)
plt.axhline(180, color='crimson', linestyle='--', alpha=0.6, label='Upper Target (180 mg/dL)')
plt.axhline(70, color='darkorange', linestyle='--', alpha=0.6, label='Lower Target (70 mg/dL)')

fight_time = pd.to_datetime('2026-08-29 01:00:00+05:30')
fight_val = h_trace[h_trace['timestamp'] == fight_time]['glucose'].values[0]
plt.scatter([fight_time], [fight_val], color='red', s=120, zorder=5)
plt.annotate('1 AM Big Fight
(210 mg/dL)', xy=(fight_time, fight_val), xytext=(fight_time - pd.Timedelta(hours=2.5), 260),
             arrowprops=dict(facecolor='red', shrink=0.08, width=1.5, headwidth=6),
             fontweight='bold', fontsize=10, bbox=dict(boxstyle='round', facecolor='#fee0d2', alpha=0.9))

max_idx = h_trace['glucose'].idxmax()
max_time = h_trace.loc[max_idx, 'timestamp']
max_val = h_trace.loc[max_idx, 'glucose']
plt.scatter([max_time], [max_val], color='purple', s=120, zorder=5)
plt.annotate(f'Morning Surge Peak:
{max_val:.1f} mg/dL (11:05 AM)', xy=(max_time, max_val), xytext=(max_time - pd.Timedelta(hours=2.5), 340),
             arrowprops=dict(facecolor='purple', shrink=0.08, width=1.5, headwidth=6),
             fontweight='bold', fontsize=10, bbox=dict(boxstyle='round', facecolor='#efedf5', alpha=0.9))

plt.title('High-Resolution CGM Trace: August 28-29 Hackathon Conflict & Morning Surge', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Time', fontsize=11)
plt.ylabel('Blood Glucose (mg/dL)', fontsize=11)
plt.legend(loc='lower left', frameon=True)
plt.show()